# LexFeed — Demo Notebook

This notebook demonstrates the LexFeed recommendation engine in action.

**What we'll show:**
1. Auto-detecting post categories with no manual input
2. Generating semantic embeddings for posts
3. Building a user interest vector from their behavior
4. Running the full 4-stage recommendation pipeline
5. Showing how the feed changes as the user interacts

In [ ]:
import sys
sys.path.append('..')

from app.core.embeddings import embed_text, cosine_similarity, average_vectors
from app.core.enrichment import enrich_post
from app.core.ranking import score_post
from app.core.diversify import diversify

print('LexFeed loaded successfully!')

## Step 1 — Auto Category Detection

A user writes a post. No dropdown. LLM reads it and classifies automatically.

In [ ]:
# Sample posts — no categories assigned by the user
sample_posts = [
    "My landlord has not returned my security deposit even after 3 months of vacating the flat.",
    "My employer deducted salary without any notice or explanation. What are my options?",
    "Supreme Court rules that digital signatures are valid for property registration in Maharashtra.",
    "How do I file an RTI to get information about a government scheme I applied for?",
    "My husband is abusive. What legal steps can I take to protect myself and my children?",
]

print('Classifying posts...')
print('-' * 60)

for i, post in enumerate(sample_posts):
    result = enrich_post(post)
    print(f"Post {i+1}: {post[:60]}...")
    print(f"  → Category: {result['primary_category']}")
    print(f"  → Urgency:  {result['urgency']}")
    print(f"  → Tags:     {', '.join(result['enriched_tags'][:3])}")
    print()

## Step 2 — Semantic Embeddings

Two posts about similar topics should have similar embeddings, even if they use different words.

In [ ]:
post_a = "Landlord refusing to return deposit after tenant vacated property."
post_b = "Owner not giving back advance payment after renter left the house."
post_c = "My employer fired me without notice or severance pay."

emb_a = embed_text(post_a)
emb_b = embed_text(post_b)
emb_c = embed_text(post_c)

sim_ab = cosine_similarity(emb_a, emb_b)
sim_ac = cosine_similarity(emb_a, emb_c)

print(f'Similarity between A and B (same topic, different words): {sim_ab:.3f}')
print(f'Similarity between A and C (different topics):            {sim_ac:.3f}')
print()
print('A and B should be much more similar than A and C.')

## Step 3 — User Interest Vector

Simulate a user who reads tenant rights and consumer posts. Their interest vector is built from those interactions.

In [ ]:
# Posts the user interacted with
liked_posts = [
    "Landlord refuses to return deposit. What are tenant rights in Maharashtra?",
    "How to send legal notice to landlord for deposit recovery?",
    "Consumer forum ruling: company must refund within 30 days.",
]

# Build interest vector by averaging embeddings of liked posts
liked_embeddings = [embed_text(p) for p in liked_posts]
user_vector = average_vectors(liked_embeddings)

print(f'User interest vector built from {len(liked_posts)} interactions.')
print(f'Vector dimension: {len(user_vector)}')
print()

# Now test: which new post is more relevant to this user?
new_post_relevant   = "Security deposit rules changed under new rental act."
new_post_irrelevant = "Corporate tax filing deadlines for Q3 2026."

sim_relevant   = cosine_similarity(embed_text(new_post_relevant), user_vector)
sim_irrelevant = cosine_similarity(embed_text(new_post_irrelevant), user_vector)

print(f'Relevance score for tenant post:    {sim_relevant:.3f}')
print(f'Relevance score for corporate post: {sim_irrelevant:.3f}')
print()
print('The tenant post should score significantly higher for this user.')

## Step 4 — Full 6-Signal Scoring

Score two posts for this user using all 6 signals.

In [ ]:
from datetime import datetime, timezone, timedelta

user_interests = {
    'interest_vector': user_vector,
    'explicit_interests': ['Tenant Rights', 'Consumer Rights'],
}

# Post 1: Relevant, recent, verified advocate
post_1 = {
    'id': 'post-001',
    'content': 'Security deposit rules changed under new rental act.',
    'type': 'Legal News',
    'primary_category': 'Tenant Rights',
    'enriched_tags': ['deposit', 'tenant', 'rental', 'maharashtra'],
    'author_verified': True,
    'author_role': 'advocate',
    'reactions_count': 24,
    'comments_count': 6,
    'created_at': (datetime.now(timezone.utc) - timedelta(hours=2)).isoformat(),
}

# Post 2: Irrelevant, old, unverified
post_2 = {
    'id': 'post-002',
    'content': 'Corporate tax filing deadlines for Q3 2026.',
    'type': 'Short Update',
    'primary_category': 'Tax Law',
    'enriched_tags': ['tax', 'corporate', 'filing'],
    'author_verified': False,
    'author_role': 'client',
    'reactions_count': 2,
    'comments_count': 0,
    'created_at': (datetime.now(timezone.utc) - timedelta(hours=36)).isoformat(),
}

emb_1 = embed_text(post_1['content'])
emb_2 = embed_text(post_2['content'])

score_1, breakdown_1 = score_post(post_1, user_interests, emb_1)
score_2, breakdown_2 = score_post(post_2, user_interests, emb_2)

print(f'Post 1 (Tenant Rights, recent, verified): {score_1}')
print(f'  Breakdown: {breakdown_1}')
print()
print(f'Post 2 (Tax Law, old, unverified): {score_2}')
print(f'  Breakdown: {breakdown_2}')
print()
print(f'Post 1 should score much higher for this user.')

## Step 5 — Diversification

Make sure the feed isn't all the same topic or author.

In [ ]:
# Simulate 10 ranked posts — 5 tenant rights, 5 from same author
ranked_posts = [
    {'id': f'p{i}', 'primary_category': 'Tenant Rights' if i < 5 else 'Family Law',
     'author_id': 'author-1' if i % 2 == 0 else 'author-2',
     'type': 'Short Update', '_score': 100 - i}
    for i in range(10)
]

diversified = diversify(ranked_posts, max_size=8)

print('Before diversification:')
for p in ranked_posts:
    print(f"  {p['id']} | {p['primary_category']} | author: {p['author_id']} | score: {p['_score']}")

print(f'\nAfter diversification ({len(diversified)} posts):')
for p in diversified:
    print(f"  {p['id']} | {p['primary_category']} | author: {p['author_id']} | score: {p['_score']}")

## Summary

LexFeed in action:
- ✅ Auto-detected post categories with no manual input
- ✅ Semantic similarity works across different wording
- ✅ User interest vector built from behavior
- ✅ 6-signal scoring gives relevant posts higher scores
- ✅ Diversification prevents repetitive feeds

**Total API cost: ₹0**